In [1]:
import polars as pl
from pathlib import Path
import fastexcel
import pandas as pd
import xlsxwriter

### Fucntion Definition

In [2]:
def read_excel(path: Path, sheet_pattern: str | None = None, had_header: bool = True) -> pl.DataFrame:
    """
    Read matching sheets from an Excel file into a single Polars DataFrame.
    
    - sheet_pattern: prefix to match sheet names (e.g. "Income" matches "Income - 1", "Income - 2").
                     None = read all sheets.
    - had_header: whether the sheet has a header row.
    """
    reader = fastexcel.read_excel(path)
    print(f"  Sheets in {path.name}: {reader.sheet_names}")

    # Filter sheets by pattern
    sheets = [s for s in reader.sheet_names
              if sheet_pattern is None or s.startswith(sheet_pattern)]
    print(f"  Matching sheets: {sheets}")

    dfs = []
    for name in sheets:
        df = pl.read_excel(path, sheet_name=name, has_header=had_header)
        df = df.with_columns(pl.lit(name).alias("_source_sheet"))
        dfs.append(df)

    if not dfs:
        return pl.DataFrame()

    combined = pl.concat(dfs, how="diagonal_relaxed") if len(dfs) > 1 else dfs[0]
    return combined.with_columns(pl.lit(path.name).alias("_source_file"))

In [ ]:
def concat_files(files: list[Path], label: str, sheet_pattern: str | int | None = None, has_header: bool = True) -> pl.DataFrame:
    if not files:
        return pl.DataFrame()
    dfs = []
    for f in files:
        df = read_excel(f, sheet_pattern=sheet_pattern, had_header=has_header)
        if label == "income_all" and not has_header:
            header = df.row(2)
            df = df.slice(3)
            df.columns = header
            dfs.append(df)
        elif label == "balance_all" and not has_header:
            header =df.row(13)
            df = df.slice(14)
            df.columns = header
            dfs.append(df)
        print(f"  Loaded {f.name}: {df.shape}")
    combined = pl.concat(dfs, how="diagonal_relaxed")
    # before = combined.height
    # Drop duplicates (exclude _source_file so rows from overlapping files are caught)
    # data_cols = [c for c in combined.columns if c != "_source_file"]
    # combined = combined.unique(subset=data_cols, keep="first")
    # after = combined.height
    # dupes = before - after
    # print(f"=> {label}: {combined.shape} (removed {dupes} duplicate rows)\n")
    return combined

### Assign File Location

In [5]:
root = Path(r"c:\Users\TanJunJie\OneDrive - SRKK Group\Project\watson_entriesmatching\OneDrive_2026-03-09\Shopee Sample Reports (Testing)\scenario2")

# Find all .xlsx files recursively
xlsx_files = sorted(root.rglob("*.xlsx"))

### Income Report

In [6]:
# --- Load & concatenate by report type ---
income_files = [f for f in xlsx_files if f.name.startswith("Income.released")]
print("=== Income Released ===")
income_all = concat_files(income_files, "income_all", sheet_pattern="Income", has_header=False)



=== Income Released ===
  Sheets in Income.released.my.20251231_20251231.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Income - 1', 'Adjustment', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20251231_20251231.xlsx: (15021, 49)
  Sheets in Income.released.my.20260101_20260101.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260101_20260101.xlsx: (12599, 49)
  Sheets in Income.released.my.20260102_20260102.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260102_20260102.xlsx: (16012, 49)
  Sheets in Income.released.my.20260103_20260103.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']


In [20]:
dup_order_id = (
income_all
.group_by("Order ID")
.len()
.filter(pl.col("len") > 1)
.height
)
print("duplicate Order ID groups:", dup_order_id)

duplicate Order ID groups: 71514


In [21]:
dup_full_rows = income_all.is_duplicated().sum()
print("exact duplicate rows:", dup_full_rows)

exact duplicate rows: 0


In [23]:
 #Select only relevant columns for matching (exclude "View By" which is used for filtering but not needed in matching)
income_all_normalized = (
    income_all.
    filter(pl.col("View By")== "Order")
    .select(
    ['Sequence No.',
    'Order ID',
    'Order Creation Date',
    'Payout Completed Date',

    'Total Released Amount (RM)',
    'Product Price',
    'Refund Amount',
    'Shipping Fee Paid by Buyer (excl. SST)',
    'Shipping Fee Charged by Logistic Provider',
    'Seller Paid Shipping Fee SST',
    'Shipping Rebate From Shopee',
    'Reverse Shipping Fee',
    'Reverse Shipping Fee SST',
    'Saver Programme Shipping Fee Savings',
    'Return to Seller Fee',
    'Rebate Provided by Shopee',
    'Voucher Sponsored by Seller',
    'Coin Cashback Sponsored by Seller',
    'Commission Fee (incl. SST)',
    'Service Fee (Incl. SST)',
    'Transaction Fee (Incl. SST)',
    'AMS Commission Fee',
    'Saver Programme Fee (Incl. SST)',

    'Username (Buyer)',
    'Transaction Fee Rate (%)',
    'Buyer Payment Method',
    'Buyer Payment Method Details_1(if applicable)',
    'Payment Details / Installment Plan',

    'Voucher Code From Seller',
    'Lost Compensation',
    ])
    .with_columns([
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars(),
        pl.col('Total Released Amount (RM)').cast(pl.Float64, strict=False),
        pl.col('Order Creation Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False),
        pl.col('Payout Completed Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False),
        pl.col('Product Price').cast(pl.Float64, strict=False),
        pl.col('Refund Amount').cast(pl.Float64, strict=False),
        pl.col('Shipping Fee Paid by Buyer (excl. SST)').cast(pl.Float64, strict=False),
        pl.col('Shipping Fee Charged by Logistic Provider').cast(pl.Float64, strict=False),
        pl.col('Seller Paid Shipping Fee SST').cast(pl.Float64, strict=False),
        pl.col('Shipping Rebate From Shopee').cast(pl.Float64, strict=False),
        pl.col('Reverse Shipping Fee').cast(pl.Float64, strict=False),
        pl.col('Reverse Shipping Fee SST').cast(pl.Float64, strict=False),
        pl.col('Saver Programme Shipping Fee Savings').cast(pl.Float64, strict=False),
        pl.col('Return to Seller Fee').cast(pl.Float64, strict=False),
        pl.col('Rebate Provided by Shopee').cast(pl.Float64, strict=False),
        pl.col('Voucher Sponsored by Seller').cast(pl.Float64, strict=False),
        pl.col('Coin Cashback Sponsored by Seller').cast(pl.Float64, strict=False),
        pl.col('Commission Fee (incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Service Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Transaction Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('AMS Commission Fee').cast(pl.Float64, strict=False),
        pl.col('Saver Programme Fee (Incl. SST)').cast(pl.Float64, strict=False),
        pl.col('Transaction Fee Rate (%)').cast(pl.Float64, strict=False),
        pl.col('Lost Compensation').cast(pl.Float64, strict=False),
    ])
    .drop_nulls(['Order ID'])
    )
income_all_normalized.head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64
"""1""","""251230QHF7YGBE""",2025-12-30,2025-12-31,-5.19,27.93,-27.93,0.0,-4.9,-0.29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""ndkshidshisbssayacantik""",3.78,"""Cash on Delivery""","""""","""""","""""",0.0
"""3""","""251230PMP6MTW6""",2025-12-30,2025-12-31,-2.65,14.9,-14.9,0.0,-2.5,-0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""gq6pe2qreq""",3.78,"""Online Banking""","""""","""""","""""",0.0
"""5""","""251230PB5YG8HT""",2025-12-30,2025-12-31,37.32,44.1,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.64,0.0,-2.14,0.0,0.0,"""fiqamokhtar""",4.86,"""SPayLater""","""""","""12x""","""""",0.0
"""7""","""251230PB3GD63E""",2025-12-30,2025-12-31,34.51,41.04,0.0,0.0,-4.9,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-4.54,0.0,-1.99,0.0,0.0,"""nurnabilahabukasim""",4.86,"""SPayLater""","""""","""3x""","""""",0.0
"""10""","""251230PAY9XQWY""",2025-12-30,2025-12-31,26.79,31.5,0.0,4.9,-4.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.32,0.0,-1.39,0.0,0.0,"""amyrahbrown""",3.78,"""Credit / Debit Card""","""""","""""","""""",0.0


### Balance Report 

In [41]:
balance_files = [f for f in xlsx_files if f.name.startswith("my_balance_transaction")]
print("=== Balance Transaction ===")
balance_all = concat_files(balance_files, "balance_all", sheet_pattern="Transaction", has_header=False)

=== Balance Transaction ===
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: (5000, 10)
  Sheets

In [42]:
balance_all.filter(pl.col("Order ID") == "2601069PFCA9QB").head()

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_5_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_6_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_7_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_8_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_9_of_9.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_1_of_8.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_2_of_8.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_3_of_8.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_4_of_8.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_5_of_8.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_6_of_8.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_7_of_8.xlsx,my_balance_transaction_report.shopee.20260107_20260114part_8_of_8.xlsx
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""2026-01-07 18:21:16""","""Order Income""","""Income from Order #2601069PFCA…","""2601069PFCA9QB""","""Money In""","""16.8""","""Transaction Completed""","""117684.9""","""Transaction Report""","""my_balance_transaction_report.…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""2026-01-07 18:21:16""","""Order Income""","""Income from Order #2601069PFCA…","""2601069PFCA9QB""","""Money In""","""16.8""","""Transaction Completed""","""117684.9""","""Transaction Report""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""my_balance_transaction_report.…",null


In [43]:
dup_order_id = (
balance_all
.group_by("Order ID")
.len()
.filter(pl.col("len") > 1)
.height
)
print("duplicate Order ID groups:", dup_order_id)

duplicate Order ID groups: 5563


In [44]:
dup_full_rows = balance_all.is_duplicated().sum()
print("exact duplicate rows:", dup_full_rows)

exact duplicate rows: 0


In [45]:
#Deduplication of entries, remove duplicates based on "Order ID" column, keep the first occurrence and maintain the original order
balance_all = balance_all.unique(
    subset=["Order ID"],
    keep="first",
    maintain_order=True
)

In [14]:
def dedupe_df(df: pl.DataFrame, name: str) -> pl.DataFrame:
    # Ignore source-tracking columns when checking duplicates
    meta_cols = [c for c in df.columns if c.startswith("_source")]
    data_cols = [c for c in df.columns if c not in meta_cols]

    before = df.height
    df_deduped = df.unique(subset=data_cols, keep="first", maintain_order=True)
    after = df_deduped.height

    print(f"{name}: {before:,} -> {after:,} (dropped {before - after:,} duplicated rows)")
    return df_deduped


balance_all = dedupe_df(balance_all, "balance_all")
income_all = dedupe_df(income_all, "income_all")
sales_report = dedupe_df(sales_report, "sales_report")

balance_all: 77,137 -> 77,137 (dropped 0 duplicated rows)
income_all: 223,297 -> 207,454 (dropped 15,843 duplicated rows)
sales_report: 62,427 -> 62,427 (dropped 0 duplicated rows)


In [46]:
balance_all.columns

['Date',
 'Transaction Type',
 'Description',
 'Order ID',
 'Money Direction',
 'Amount',
 'Status',
 'Balance After Transactions',
 'Transaction Report',
 'my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_5_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_6_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_7_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_8_of_9.xlsx',
 'my_balance_transaction_report.shopee.20251231_20260107part_9_of_9.xlsx',
 'my_balance_transaction_report.shopee.20260107_20260114part_1_of_8.xlsx',
 'my_balance_transaction_report.shopee.20260107_20260114part_2_of_8.xlsx',
 'my_balance_transac

In [49]:
balance_all_normalized = (
    balance_all
    .select(['Date',
        'Transaction Type',
        'Description',
        'Order ID',
        'Money Direction',
        'Amount',
        'Status',
        'Balance After Transactions',
        'Transaction Report'])
    # Step 1: parse/cast raw types first
    .with_columns([
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars(),
        pl.col('Date').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S'),
        pl.col('Amount').cast(pl.Float64, strict=False).alias('Payment Amount'),
        pl.col('Balance After Transactions').cast(pl.Float64, strict=False),
    ])
    # Step 2: derive columns from the now-Datetime "Date"
    .with_columns([
        pl.col('Date').dt.strftime('%d-%b-%Y').str.to_uppercase().alias('Payment Date'),
        pl.col('Date').dt.strftime('%b-%Y').str.to_uppercase().alias('Payment Month'),
    ])
)
balance_all_normalized.head(10)

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,Payment Amount,Payment Date,Payment Month
datetime[μs],str,str,str,str,str,str,f64,str,f64,str,str
2026-01-07 23:59:53,"""Order Income""","""Income from Order #251227FMQA3…","""251227FMQA3XGP""","""Money In""","""13.42""","""Transaction Completed""",195391.02,"""Transaction Report""",13.42,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:46,"""Order Income""","""Income from Order #260102VN9VM…","""260102VN9VMVPV""","""Money In""","""8.31""","""Transaction Completed""",195377.6,"""Transaction Report""",8.31,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:32,"""Order Income""","""Income from Order #2601045P6TK…","""2601045P6TK6PG""","""Money In""","""21.25""","""Transaction Completed""",195369.29,"""Transaction Report""",21.25,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:28,"""Order Income""","""Income from Order #2601045Y4YT…","""2601045Y4YTEG6""","""Money In""","""13.19""","""Transaction Completed""",195348.04,"""Transaction Report""",13.19,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:26,"""Order Income""","""Income from Order #2601045E6UG…","""2601045E6UGTNU""","""Money In""","""23.08""","""Transaction Completed""",195334.85,"""Transaction Report""",23.08,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:06,"""Order Income""","""Income from Order #26010575HF5…","""26010575HF5Q6D""","""Money In""","""14.92""","""Transaction Completed""",195311.77,"""Transaction Report""",14.92,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:59:02,"""Order Income""","""Income from Order #26010347C47…","""26010347C470NX""","""Money In""","""16.71""","""Transaction Completed""",195296.85,"""Transaction Report""",16.71,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:58:33,"""Order Income""","""Income from Order #260101UAHS8…","""260101UAHS8FRR""","""Money In""","""6.76""","""Transaction Completed""",195280.14,"""Transaction Report""",6.76,"""07-JAN-2026""","""JAN-2026"""
2026-01-07 23:58:22,"""Order Income""","""Income from Order #2601069VSE7…","""2601069VSE7UX0""","""Money In""","""33.34""","""Transaction Completed""",195273.38,"""Transaction Report""",33.34,"""07-JAN-2026""","""JAN-2026"""


In [39]:
balance_all_normalized.head()

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report
datetime[μs],str,str,str,str,f64,str,f64,str
2026-01-07 23:59:53,"""Order Income""","""Income from Order #251227FMQA3…","""251227FMQA3XGP""","""Money In""",13.42,"""Transaction Completed""",195391.02,"""Transaction Report"""
2026-01-07 23:59:46,"""Order Income""","""Income from Order #260102VN9VM…","""260102VN9VMVPV""","""Money In""",8.31,"""Transaction Completed""",195377.6,"""Transaction Report"""
2026-01-07 23:59:32,"""Order Income""","""Income from Order #2601045P6TK…","""2601045P6TK6PG""","""Money In""",21.25,"""Transaction Completed""",195369.29,"""Transaction Report"""
2026-01-07 23:59:28,"""Order Income""","""Income from Order #2601045Y4YT…","""2601045Y4YTEG6""","""Money In""",13.19,"""Transaction Completed""",195348.04,"""Transaction Report"""
2026-01-07 23:59:26,"""Order Income""","""Income from Order #2601045E6UG…","""2601045E6UGTNU""","""Money In""",23.08,"""Transaction Completed""",195334.85,"""Transaction Report"""


### Sales Report

In [28]:
sales_files = sorted(root.parent.glob("SalesReport*.xlsx"))
print("=== Sales Reports ===")
print([f.name for f in sales_files])

sales_dfs = []
for f in sales_files:
    df = pl.read_excel(f, sheet_name="SalesReport", has_header=True)
    df = df.with_columns(pl.lit(f.name).alias("_source_file"))
    sales_dfs.append(df)

sales_report = pl.concat(sales_dfs, how="diagonal_relaxed") if sales_dfs else pl.DataFrame()
print("sales_report shape:", sales_report.shape)

sales_report.head()

=== Sales Reports ===
['SalesReport Wk2.xlsx', 'SalesReport Wk3.xlsx']


Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string


sales_report shape: (62427, 10)


OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file
str,str,i64,i64,date,f64,str,str,str,str
"""9524949786""","""2601044PUM2781""",233603239,895,2026-01-05,21.0,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524950020""","""26010450MFPQE1""",233603241,895,2026-01-05,98.29,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524951072""","""2601045R7NTP28""",233603245,895,2026-01-05,107.65,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524928466""","""260101TUY6HNTV""",233603250,895,2026-01-05,9.56,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524930857""","""260101UF58WU38""",233603253,895,2026-01-05,2.73,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""


In [38]:
# Normalize sales report
sales_report_normalized = (
    sales_report
    .with_columns([
        # `SalesWorkDate` may already be a Date/Datetime (from Excel reader),
        # or it may be a string like "31-12-2025"; handle both safely.
        pl.coalesce([
            pl.col("SalesWorkDate").cast(pl.Date, strict=False),
            pl.col("SalesWorkDate")
              .cast(pl.Utf8, strict=False)
              .str.strip_chars()
              .str.strptime(pl.Date, "%d-%m-%Y", strict=False),
        ]).alias("WorkDate"),
        pl.col("MarketPlaceOrderNum").cast(pl.Utf8).str.strip_chars().alias("Order ID"),
        pl.col("TotalAmount").cast(pl.Float64, strict=False).alias("SalesCenterAmount"),
        pl.col("WorkDate").dt.strftime("%b-%Y").str.to_uppercase().alias("Sales Month")
    ])
)
sales_report_normalized.head()

OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file,WorkDate,Order ID,SalesCenterAmount,Sales Month
str,str,i64,i64,date,f64,str,str,str,str,date,str,f64,str
"""9524949786""","""2601044PUM2781""",233603239,895,2026-01-05,21.0,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""2601044PUM2781""",21.0,"""Jan-2026"""
"""9524950020""","""26010450MFPQE1""",233603241,895,2026-01-05,98.29,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""26010450MFPQE1""",98.29,"""Jan-2026"""
"""9524951072""","""2601045R7NTP28""",233603245,895,2026-01-05,107.65,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""2601045R7NTP28""",107.65,"""Jan-2026"""
"""9524928466""","""260101TUY6HNTV""",233603250,895,2026-01-05,9.56,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""260101TUY6HNTV""",9.56,"""Jan-2026"""
"""9524930857""","""260101UF58WU38""",233603253,895,2026-01-05,2.73,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx""",2026-01-05,"""260101UF58WU38""",2.73,"""Jan-2026"""


### Generate Report

In [35]:
first_part = sales_report_normalized.select(["Order ID", "OrderNum","MarketPlaceOrderNum", "SalesWorkDate","SalesCenterAmount","Sales Month"])
first_part.head()

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month
str,str,str,date,f64,str
"""2601044PUM2781""","""9524949786""","""2601044PUM2781""",2026-01-05,21.0,"""JAN-2026"""
"""26010450MFPQE1""","""9524950020""","""26010450MFPQE1""",2026-01-05,98.29,"""JAN-2026"""
"""2601045R7NTP28""","""9524951072""","""2601045R7NTP28""",2026-01-05,107.65,"""JAN-2026"""
"""260101TUY6HNTV""","""9524928466""","""260101TUY6HNTV""",2026-01-05,9.56,"""JAN-2026"""
"""260101UF58WU38""","""9524930857""","""260101UF58WU38""",2026-01-05,2.73,"""JAN-2026"""


In [50]:
payment_date = balance_all_normalized.select(["Order ID", "Payment Date", "Payment Month", "Payment Amount"])
payment_date.head()

Order ID,Payment Date,Payment Month,Payment Amount
str,str,str,f64
"""251227FMQA3XGP""","""07-JAN-2026""","""JAN-2026""",13.42
"""260102VN9VMVPV""","""07-JAN-2026""","""JAN-2026""",8.31
"""2601045P6TK6PG""","""07-JAN-2026""","""JAN-2026""",21.25
"""2601045Y4YTEG6""","""07-JAN-2026""","""JAN-2026""",13.19
"""2601045E6UGTNU""","""07-JAN-2026""","""JAN-2026""",23.08


In [56]:
income_all_normalized.head(2)
income_all_normalized.columns

['Sequence No.',
 'Order ID',
 'Order Creation Date',
 'Payout Completed Date',
 'Total Released Amount (RM)',
 'Product Price',
 'Refund Amount',
 'Shipping Fee Paid by Buyer (excl. SST)',
 'Shipping Fee Charged by Logistic Provider',
 'Seller Paid Shipping Fee SST',
 'Shipping Rebate From Shopee',
 'Reverse Shipping Fee',
 'Reverse Shipping Fee SST',
 'Saver Programme Shipping Fee Savings',
 'Return to Seller Fee',
 'Rebate Provided by Shopee',
 'Voucher Sponsored by Seller',
 'Coin Cashback Sponsored by Seller',
 'Commission Fee (incl. SST)',
 'Service Fee (Incl. SST)',
 'Transaction Fee (Incl. SST)',
 'AMS Commission Fee',
 'Saver Programme Fee (Incl. SST)',
 'Username (Buyer)',
 'Transaction Fee Rate (%)',
 'Buyer Payment Method',
 'Buyer Payment Method Details_1(if applicable)',
 'Payment Details / Installment Plan',
 'Voucher Code From Seller',
 'Lost Compensation']

In [83]:
thrid_part = income_all_normalized.select(
    "Order ID",
    pl.col("Commission Fee (incl. SST)").alias("Commission Fee"),
    pl.col("Transaction Fee (Incl. SST)").alias("Transaction Fee"),
    pl.col("Service Fee (Incl. SST)").alias("Service Fee"),
    pl.col("Return to Seller Fee").alias("Return QC Fee"),
    pl.col("Rebate Provided by Shopee").alias("Voucher/(Disc rebate)"),
    pl.col("Refund Amount").alias("Refund"),
    (pl.col('Shipping Fee Paid by Buyer (excl. SST)') + pl.col('Shipping Fee Charged by Logistic Provider')+pl.col('Seller Paid Shipping Fee SST') + pl.col('Shipping Rebate From Shopee') + pl.col('Reverse Shipping Fee') + pl.col('Reverse Shipping Fee SST')).alias("Actual Shipping Fee")
)

In [85]:
recon_report = first_part.join(payment_date, on=["Order ID"], how="left") \
                         .join(thrid_part, on=["Order ID"], how="left")
recon_report.columns

['Order ID',
 'OrderNum',
 'MarketPlaceOrderNum',
 'SalesWorkDate',
 'SalesCenterAmount',
 'Sales Month',
 'Payment Date',
 'Payment Month',
 'Payment Amount',
 'Commission Fee',
 'Transaction Fee',
 'Service Fee',
 'Return QC Fee',
 'Voucher/(Disc rebate)',
 'Refund',
 'Actual Shipping Fee']

In [86]:
recon_report = recon_report.with_columns(
    (pl.col("SalesCenterAmount") - pl.col("Payment Amount") + pl.col("Commission Fee") 
     + pl.col("Transaction Fee") + pl.col("Service Fee") + pl.col("Return QC Fee") + pl.col("Voucher/(Disc rebate)") + pl.col("Actual Shipping Fee")).alias("Outstanding")
)
recon_report.head()

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month,Payment Date,Payment Month,Payment Amount,Commission Fee,Transaction Fee,Service Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Actual Shipping Fee,Outstanding
str,str,str,date,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2601044PUM2781""","""9524949786""","""2601044PUM2781""",2026-01-05,21.0,"""JAN-2026""",null,null,null,null,null,null,null,null,null,null,null
"""26010450MFPQE1""","""9524950020""","""26010450MFPQE1""",2026-01-05,98.29,"""JAN-2026""",null,null,null,null,null,null,null,null,null,null,null
"""2601045R7NTP28""","""9524951072""","""2601045R7NTP28""",2026-01-05,107.65,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",91.35,-11.07,-5.23,0.0,0.0,0.0,0.0,0.0,1.0658e-14
"""260101TUY6HNTV""","""9524928466""","""260101TUY6HNTV""",2026-01-05,9.56,"""JAN-2026""",null,null,null,null,null,null,null,null,null,null,null
"""260101UF58WU38""","""9524930857""","""260101UF58WU38""",2026-01-05,2.73,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",2.33,-0.3,-0.1,0.0,0.0,0.0,0.0,0.0,-8.3267e-17


In [72]:
recon_report.filter(pl.col("Order ID") == "2601045R7NTP28")

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month,Payment Date,Payment Month,Payment Amount,Commission Fee,Transaction Fee,Service Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Outstanding
str,str,str,date,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""2601045R7NTP28""","""9524951072""","""2601045R7NTP28""",2026-01-05,107.65,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",91.35,-11.07,-5.23,0.0,0.0,0.0,0.0,1.0658e-14


In [73]:
recon_report.filter(pl.col("Order ID") == "260101UF58WU38")

Order ID,OrderNum,MarketPlaceOrderNum,SalesWorkDate,SalesCenterAmount,Sales Month,Payment Date,Payment Month,Payment Amount,Commission Fee,Transaction Fee,Service Fee,Return QC Fee,Voucher/(Disc rebate),Refund,Outstanding
str,str,str,date,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""260101UF58WU38""","""9524930857""","""260101UF58WU38""",2026-01-05,2.73,"""JAN-2026""","""05-JAN-2026""","""JAN-2026""",2.33,-0.3,-0.1,0.0,0.0,0.0,0.0,-8.3267e-17


In [76]:
path = './Shopee Sample Reports (Testing)/Pivot table-Shopee 180126.xlsx'
pivot = pl.read_excel(path, sheet_name="Payment Summary-pivot", has_header=True)
pivot.head()

Could not determine dtype for column 9, falling back to string


Row Labels,Sum of Total Released Amount (RM),Sum of Voucher Sponsored by Seller,Sum of Rebate Provided by Shopee,Sum of Refund Amount,Sum of Commission Fee (incl. SST),Sum of Transaction Fee (Incl. SST),Sum of Service Fee (Incl. SST),Sum of AMS Commission Fee,Sum of Return QC Fee
str,f64,f64,f64,f64,f64,f64,f64,f64,str
"""-""",164036.71,null,null,null,null,null,null,null,null
"""(blank)""",null,null,null,null,null,null,null,null,null
"""250712VX0ASXUY""",83.22,-10.0,0.0,0.0,-10.55,-3.74,0.0,0.0,null
"""2507120FDF5NFV""",11.37,0.0,0.0,0.0,-1.52,-0.51,0.0,0.0,null
"""250705B8MM4UH4""",23.45,0.0,0.0,0.0,-2.87,-1.61,0.0,0.0,null


In [78]:
pivot.filter(pl.col("Sum of Return QC Fee").is_not_null())

Row Labels,Sum of Total Released Amount (RM),Sum of Voucher Sponsored by Seller,Sum of Rebate Provided by Shopee,Sum of Refund Amount,Sum of Commission Fee (incl. SST),Sum of Transaction Fee (Incl. SST),Sum of Service Fee (Incl. SST),Sum of AMS Commission Fee,Sum of Return QC Fee
str,f64,f64,f64,f64,f64,f64,f64,f64,str


In [79]:
income_all_normalized.filter(pl.col("Order ID") == "260110P1CNKDKW").head()

Sequence No.,Order ID,Order Creation Date,Payout Completed Date,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Voucher Code From Seller,Lost Compensation
str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,str,f64
"""600""","""260110P1CNKDKW""",2026-01-10,2026-01-12,124.06,156.96,0.0,9.0,-10.0,-0.06,0.0,0.0,0.0,0.0,0.0,0.0,-15.0,0.0,-11.11,0.0,-5.73,0.0,0.0,"""sy4visk8tr""",3.78,"""Cash on Delivery""","""""","""""","""WATS0110B""",0.0
